In [3]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import math

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

arquivo = (
    "/Users/claudiojacomassi/Documents/GitHub/"
    "Suicidio_adolescentes_Brasil/"
    "02_Tratamento_dados_database_PENSE_2024/"
    "02_Scripts_tratamento_dados_PENSE_2024/"
    "07_PENSE2024_IDADE_13_A_17_CATEGORY.parquet"
)

# Salvar na pasta de trabalho atual
pasta_graficos = Path.cwd() / "03_Graficos_frequencia"

pasta_graficos.mkdir(parents=True, exist_ok=True)

pasta_graficos.mkdir(parents=True, exist_ok=True)

# Aparência dos gráficos
plt.style.use("ggplot")

# Número máximo de categorias exibidas por gráfico.
# Use None para mostrar todas as categorias.
max_categorias = 30

# ============================================================
# 2. LER A BASE DE DADOS
# ============================================================

df = pd.read_parquet(arquivo)

print(f"Arquivo: {arquivo}")
print(f"Número de linhas: {len(df):,}")
print(f"Número de variáveis: {len(df.columns)}")

# Garantir que todas as variáveis sejam categóricas
df = df.astype("category")

# ============================================================
# 3. FUNÇÃO PARA GERAR GRÁFICO DE FREQUÊNCIA
# ============================================================

def gerar_grafico_frequencia(df, variavel, pasta, max_categorias=30):

    serie = df[variavel]

    # Frequência absoluta, incluindo valores ausentes
    frequencia = serie.value_counts(
        dropna=False,
        sort=True
    )

    # Identificar valores ausentes para exibição
    frequencia.index = [
        "(Ausente)" if pd.isna(valor) else str(valor)
        for valor in frequencia.index
    ]

    # Agrupar categorias menos frequentes, se necessário
    if max_categorias is not None and len(frequencia) > max_categorias:
        principais = frequencia.iloc[:max_categorias]
        outras = frequencia.iloc[max_categorias:].sum()

        frequencia = principais.copy()
        frequencia.loc["Outras categorias"] = outras

    # Percentual sobre o total de linhas
    percentual = frequencia / len(df) * 100

    # Ordenar para facilitar a leitura horizontal
    frequencia = frequencia.sort_values(ascending=True)
    percentual = percentual.reindex(frequencia.index)

    # Altura dinâmica conforme o número de categorias
    altura = max(5, len(frequencia) * 0.35)

    fig, ax = plt.subplots(figsize=(12, altura))

    barras = ax.barh(
        frequencia.index.astype(str),
        frequencia.values
    )

    # Rótulos com frequência absoluta e percentual
    maior_frequencia = frequencia.max() if len(frequencia) else 0
    margem = max(maior_frequencia * 0.015, 1)

    for barra, categoria in zip(barras, frequencia.index):
        n = frequencia.loc[categoria]
        pct = percentual.loc[categoria]

        ax.text(
            barra.get_width() + margem,
            barra.get_y() + barra.get_height() / 2,
            f"{n:,.0f} ({pct:.1f}%)",
            va="center",
            fontsize=9
        )

    ax.set_title(
        f"Distribuição de frequência: {variavel}",
        fontsize=14,
        pad=15
    )
    ax.set_xlabel("Frequência absoluta")
    ax.set_ylabel("Categoria")

    ax.set_xlim(0, maior_frequencia * 1.25 if maior_frequencia else 1)
    ax.grid(axis="x", alpha=0.3)
    ax.grid(axis="y", visible=False)

    plt.tight_layout()

    # Nome seguro para salvar o arquivo
    nome_arquivo = "".join(
        c if c.isalnum() or c in "-_ " else "_"
        for c in str(variavel)
    ).strip().replace(" ", "_")

    fig.savefig(
        pasta / f"{nome_arquivo}.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# 4. GERAR UM GRÁFICO PARA CADA VARIÁVEL
# ============================================================

for i, variavel in enumerate(df.columns, start=1):

    gerar_grafico_frequencia(
        df=df,
        variavel=variavel,
        pasta=pasta_graficos,
        max_categorias=max_categorias
    )

    if i % 10 == 0 or i == len(df.columns):
        print(f"Gráficos gerados: {i}/{len(df.columns)}")

print("\nProcesso concluído!")
print(f"Gráficos salvos em: {pasta_graficos}")


Arquivo: /Users/claudiojacomassi/Documents/GitHub/Suicidio_adolescentes_Brasil/02_Tratamento_dados_database_PENSE_2024/02_Scripts_tratamento_dados_PENSE_2024/07_PENSE2024_IDADE_13_A_17_CATEGORY.parquet
Número de linhas: 118,036
Número de variáveis: 39
Gráficos gerados: 10/39
Gráficos gerados: 20/39
Gráficos gerados: 30/39
Gráficos gerados: 39/39

Processo concluído!
Gráficos salvos em: /Users/claudiojacomassi/Documents/GitHub/Suicidio_adolescentes_Brasil/03_Analise_estatistica/03_Graficos_frequencia
